In [ ]:
from jax import config as jax_config

import matplotlib.pyplot as plt

import colors
import helpers_builders
import plot_funcs
import importlib
import numpy as np
from config import CFG
from EquilibriumClass import EquilibriumClass
from StateClass import StateClass
from SupervisorClass import SupervisorClass
from VariablesClass import VariablesClass

jax_config.update("jax_enable_x64", True)

In [ ]:
# ------ Roie's color list ------
colors_lst, red, custom_cmap = colors.color_scheme()
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=colors_lst)

## Configured model

All user-editable parameters are in `config.py`.

In [ ]:
import config
importlib.reload(config)
from config import CFG

import VariablesClass
importlib.reload(VariablesClass)
from VariablesClass import VariablesClass

import SupervisorClass
importlib.reload(SupervisorClass)
from SupervisorClass import SupervisorClass

import EquilibriumClass
importlib.reload(EquilibriumClass)
from EquilibriumClass import EquilibriumClass

import StateClass
importlib.reload(StateClass)
from StateClass import StateClass

# ------ initialize class instances: user variables, supervisor controlling pulses, 
#        system bit state and Equilibrium solver using ODEs                        ------
Variabs = VariablesClass(CFG, plot_potential=CFG.Output.plot_potential)
Sprvsr = SupervisorClass(CFG)
State = StateClass(CFG, Variabs)
Eq = EquilibriumClass(Variabs)

## Input pulse

In [ ]:
impulse_dyn = Sprvsr.program_impulse()
plot_funcs.plot_impulse(Sprvsr.timepoints, impulse_dyn)

## Training

In [ ]:
initial_state = CFG.Sprvsr.initial_states[2]
State.set_state(Variabs, initial_state)
Sprvsr.set_desired_state()

for t in range(Sprvsr.T):
    # ------ measurement ------
    Sprvsr.measure(State)

    # ------ update ------
    Sprvsr.calc_loss(t)
    for amplitude in Sprvsr.calc_update_vals(t):
        Sprvsr.program_impulse(amplitude=amplitude)
        Eq.solve(State, Sprvsr)
    State.measure_state(t)

In [ ]:
importlib.reload(plot_funcs)
# pulse_amplitudes = Sprvsr.calc_update_vals(1)
fig, axes = plot_funcs.plot_training(
    Sprvsr.update_A_in_t,
    State.state_in_t,
    Sprvsr.loss_in_t,
)

## All-to-all study

In [ ]:
importlib.reload(plot_funcs)   

initial_states = ("000", "001", "010", "011", "100", "101", "110", "111")
desired_states = ("000", "001", "010", "011", "100", "101", "110", "111")
success_t_BEASTAL_short = []

for i, initial_state in enumerate(initial_states):
    for j, desired_state in enumerate(desired_states[i+1:]):
        Sprvsr = SupervisorClass(CFG)
        State = StateClass(CFG, Variabs)
        State.set_state(Variabs, initial_state)
        Sprvsr.set_desired_state(desired_state)

        for t in range(Sprvsr.T):
            # ------ measurement ------
            Sprvsr.measure(State)

            # ------ update ------
            Sprvsr.calc_loss(t)
            if Sprvsr.loss == 0:
                success_t_BEASTAL_short.append(t)
                print(r'success after {} steps'.format(t))
                break
            for amplitude in Sprvsr.calc_update_vals(t):
                Sprvsr.program_impulse(amplitude=amplitude)
                Eq.solve(State, Sprvsr)
            State.measure_state(t)
            
        fig, axes = plot_funcs.plot_training(Sprvsr.update_A_in_t, State.state_in_t, Sprvsr.loss_in_t)
        plt.show()

In [ ]:
Sprvsr.update_A_in_t

In [ ]:
Sprvsr.loss == 0

In [ ]:
np.mean(success_t_BEASTAL_short)

In [ ]:
sys.exit("completed training")

## Simulate impulses

In [ ]:
importlib.reload(plot_funcs)
importlib.reload(helpers_builders)

# simulate the same pulse from each initial state
dyn_by_state = {}

# loop initial state
for initial_state in Sprvsr.initial_states:
    State.set_state(Variabs, initial_state)
    Eq.solve(State, Sprvsr)
    dyn_by_state[initial_state] = {"u_dyn": Eq.u_dyn, "delta_dyn": Eq.delta_dyn, "F_dyn": Eq.F_dyn}
    if CFG.Output.plot_responses:  # optionally plot
        fig, axes = plot_funcs.plot_response(Eq.u_dyn, Eq.delta_dyn, Eq.F_dyn, Sprvsr.timepoints, Sprvsr.impulse_dyn, sup_title=f"Initial state {initial_state}")
        transform_fig, transform_axes = plot_funcs.plot_laplace_fourier(Eq.F_dyn, Sprvsr.timepoints, sup_title=f"Initial state {initial_state}", laplace_sigma=1)

In [ ]:
np.shape(Eq.u_dyn), np.shape(Eq.delta_dyn)

## State and endpoint-force comparison

In [ ]:
# ------ plot all states, forces and optionally their FFTs in time ------
importlib.reload(plot_funcs)

for initial_state, dyn in dyn_by_state.items():
    final_state = State.get_system_state(dyn["delta_dyn"][-1], threshold=CFG.Output.state_identification_threshold)
    print(f"Initial {initial_state} -> final state: {final_state} " f"({helpers_builders.state_to_number(final_state)})")

if CFG.Output.compare_endpoint_forces and len(dyn_by_state) == 2:
    F_dyn_by_state = {state: dyn["F_dyn"] for state, dyn in dyn_by_state.items()}
    _, _, force_delay = plot_funcs.plot_force_comparison(F_dyn_by_state, Sprvsr.timepoints, Sprvsr.start_time, 
                                                         CFG.Output.force_arrival_threshold_fraction, show_transform=True)
    print(f"Force-arrival delay: {force_delay * 1e3:.1f} ms")

## Laplace and FFT for all eight initial states
Rows contain 000; 001, 010, 100; 011, 101, 110; and 111. Run the simulations for all eight states first.

In [ ]:
state_rows = (("000",), ("001", "010", "100"), ("011", "101", "110"), ("111",))
missing_states = [state for row in state_rows for state in row if state not in dyn_by_state]
if missing_states:
    raise ValueError(f"Run the simulations with all eight initial_states in config.py first. Missing: {', '.join(missing_states)}")

laplace_sigma = 0.0  # s^-1, matching the individual-state plots
laplace_states_fig, laplace_states_axes = plt.subplots(4, 3, figsize=(15, 13), sharex=True, sharey=True, layout="constrained")
fft_states_fig, fft_states_axes = plt.subplots(4, 3, figsize=(15, 13), sharex=True, sharey=True, layout="constrained")
laplace_states_fig.suptitle(fr"Endpoint Force Laplace — all initial states ($\sigma={laplace_sigma:g}$ s$^{{-1}}$)", fontsize=16)
fft_states_fig.suptitle("Endpoint Force FFT — all initial states", fontsize=16)
for axes in (laplace_states_axes, fft_states_axes):
    for row in (0, 3):
        for column in (0, 2):
            axes[row, column].set_visible(False)
for row_index, states in enumerate(state_rows):
    for column_index, state in zip((1,) if len(states) == 1 else range(3), states):
        laplace_ax, fft_ax = laplace_states_axes[row_index, column_index], fft_states_axes[row_index, column_index]
        F_dyn = np.asarray(dyn_by_state[state]["F_dyn"])
        for force_dyn, color, label in zip((F_dyn[:, 0], F_dyn[:, -1]), colors_lst[:2], ("First truss", "Final truss")):
            frequencies, spectrum = helpers_builders.force_fft(Sprvsr.timepoints, force_dyn)
            nonnegative = frequencies >= 0
            frequencies, spectrum = frequencies[nonnegative], spectrum[nonnegative]
            _, laplace = helpers_builders.force_laplace(Sprvsr.timepoints, force_dyn, laplace_sigma + 2j * np.pi * frequencies)
            laplace_ax.plot(frequencies, np.abs(laplace), color=color, label=label)
            fft_ax.plot(frequencies, spectrum.real, color=color, linestyle="-", label=f"{label} (real)")
            fft_ax.plot(frequencies, spectrum.imag, color=color, linestyle=":", label=f"{label} (imaginary)")
        laplace_ax.set_ylabel("Transform magnitude (N s)")
        fft_ax.set_ylabel("Transform (N s)")
        for ax in (laplace_ax, fft_ax):
            ax.set_title(state, fontsize=14)
            ax.set_xlabel("Frequency (Hz)")
            ax.set_xlim(0, float(frequencies[-1]) / 2 if frequencies[-1] > 0 else 1.0)
            ax.tick_params(labelbottom=True, labelleft=True)
            ax.legend(fontsize=8)
            ax.grid(False)
plt.show()

## Amplitude study

In [ ]:
# Quick amplitude study: every initial buckle configuration, one sine cycle at 23.3 Hz
importlib.reload(helpers_builders)
from copy import copy
from matplotlib.colors import BoundaryNorm, ListedColormap

A_values_mm = np.linspace(-30.0, 30.0, 80)  # 1 mm steps, including zero
# f_values_hz = np.array([20.0])
f_values_hz = np.linspace(20.0, 30.0, 20)
study_states = tuple(format(number, f'0{Variabs.n_physical_units}b') for number in range(2 ** Variabs.n_physical_units))
study_supervisor = copy(Sprvsr)
study_supervisor.timepoints = Sprvsr.timepoints
study_state_instance = StateClass(CFG, Variabs)
study_eq = EquilibriumClass(Variabs)

def simulate_amplitude_final_state(A_mm: float, f_hz: float, initial_state: str) -> str:
    """Return the final state after one sine cycle from the initialized study state."""
    study_state_instance.set_state(Variabs, initial_state)
    study_supervisor.program_impulse(impulse_type='single_sine_cycle', amplitude=A_mm * 1e-3, frequency=f_hz)
    study_eq.solve(study_state_instance, study_supervisor)
    if not np.all(np.isfinite(study_state_instance.delta)):
        raise RuntimeError(f'Nonfinite final displacement at A={A_mm:g} mm, f={f_hz:g} Hz.')
    return study_state_instance.get_system_state(study_state_instance.delta, threshold=CFG.Output.state_identification_threshold)

parameter_states_by_state, state_number_grids_by_state = {}, {}
for initial_study_state in study_states[:2]:
    parameter_states = [(float(A_mm), float(f_hz), simulate_amplitude_final_state(float(A_mm), float(f_hz), initial_study_state)) for A_mm in A_values_mm for f_hz in f_values_hz]
    parameter_states_by_state[initial_study_state] = parameter_states
    state_number_grids_by_state[initial_study_state] = np.array([helpers_builders.state_to_number(row[2]) for row in parameter_states]).reshape(len(A_values_mm), len(f_values_hz))
    print(f'Initial {initial_study_state}: completed {len(parameter_states)} amplitude simulations')

In [ ]:
# Final-state maps, sharing the same discrete colors for all initial states
plot_only_simulated_states = True  # Leave panels for unsimulated initial states blank
state_labels = list(study_states)
_, _, study_cmap = colors.color_scheme()
state_cmap = ListedColormap(study_cmap(np.linspace(0, 1, len(state_labels))))
state_norm = BoundaryNorm(np.arange(-0.5, len(state_labels) + 0.5), state_cmap.N)
A_edges_mm = np.concatenate(([A_values_mm[0] - (A_values_mm[1] - A_values_mm[0]) / 2], (A_values_mm[:-1] + A_values_mm[1:]) / 2, [A_values_mm[-1] + (A_values_mm[-1] - A_values_mm[-2]) / 2]))
f_edges_hz = f_values_hz[0] + np.array([-0.5, 0.5]) if len(f_values_hz) == 1 else np.concatenate(([f_values_hz[0] - (f_values_hz[1] - f_values_hz[0]) / 2], (f_values_hz[:-1] + f_values_hz[1:]) / 2, [f_values_hz[-1] + (f_values_hz[-1] - f_values_hz[-2]) / 2]))
study_ncols = min(4, len(study_states))
study_nrows = (len(study_states) + study_ncols - 1) // study_ncols
amplitude_fig, amplitude_axes = plt.subplots(study_nrows, study_ncols, figsize=(3 * study_ncols, 4 * study_nrows), sharex=True, sharey=True, squeeze=False, layout='constrained')
plotted_axes, state_map = [], None
for ax, study_state in zip(amplitude_axes.flat, study_states):
    if study_state not in state_number_grids_by_state:
        if not plot_only_simulated_states:
            raise ValueError(f'Initial state {study_state} has not been simulated.')
        ax.set_visible(False)
        continue
    state_map = ax.pcolormesh(f_edges_hz, A_edges_mm, state_number_grids_by_state[study_state], cmap=state_cmap, norm=state_norm, shading='flat')
    plotted_axes.append(ax)
    ax.set(xlabel='Frequency (Hz)', ylabel='Amplitude (mm)', title=f'Initial {study_state}', xticks=f_values_hz, yticks=np.linspace(A_values_mm[0], A_values_mm[-1], 5), ylim=(A_edges_mm[0], A_edges_mm[-1]))
    ax.tick_params(labelbottom=True, labelleft=True)
    ax.grid(False)
for ax in list(amplitude_axes.flat)[len(study_states):]:
    ax.set_visible(False)
if state_map is None:
    raise ValueError('No simulated initial states are available to plot.')
state_colorbar = amplitude_fig.colorbar(state_map, ax=plotted_axes, ticks=range(len(state_labels)), label='Final state', pad=0.02)
state_colorbar.ax.set_yticklabels(state_labels)
frequency_title = f'at {f_values_hz[0]:g} Hz' if len(f_values_hz) == 1 else f'from {f_values_hz[0]:g} to {f_values_hz[-1]:g} Hz'
amplitude_fig.suptitle(f'Final state after one sine-cycle input {frequency_title}', fontsize=16)
plt.show()